# Python, part 3 · pandas

**Goal:** write course 1's `strip_and_null` yourself, the first step of session 2, and use it to answer the Trust Test's Q1.

You'll learn how pandas holds a table, how to ask a question of a whole column at once, and how to filter rows. It's the same Python as parts 1 and 2, done to 100,000 values at a time.

**How to use this notebook** (the same as course 1)
- Start the lab cluster first (**Compute**; a cold start takes 5 minutes or more), then attach this notebook to it.
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = "P3"

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
import pandas as pd
from lnl import check, cleaning, inside, pipeline
from lnl.check import ___

## 1. Remember Q1?
The Trust Test's Q1 asked: *how many records are missing a custody level?* Here are the most common values in that column, as written. `.tolist()` turns them into a list, so the quotes show where each one starts and ends:

In [ ]:
raw = pipeline.load_raw()
custody = raw["custody_level"]
print(custody.value_counts().head(8).index.tolist())

After the five real levels come values that look empty: `''`, then `'  '`, `'       '`... Some are nothing at all. Many more are only spaces. All of them mean the custody level is missing. By the end of this notebook you'll have counted them both ways, and written the function that course 1 used to fix them.

## 2. A table and its columns
pandas holds a table as a **DataFrame**. `raw` is one. Each column is a **Series**, a list of values with row numbers. You pick a column with its name in square brackets, the same as looking a value up in a dictionary in part 2:

In [ ]:
print(raw.shape)              # (rows, columns)
print(raw.columns.tolist()[:6])  # the first six column names
custody.head()

## 3. One question, asked of every value
In part 2 you asked a question of one value at a time, in a loop. pandas asks it of the whole column at once and hands back a column of `True` and `False`:

In [ ]:
(custody == "G2").head()

`.sum()` adds them up, and `True` counts as 1, so the sum is how many are `True`:

In [ ]:
print((custody == "G2").sum())

That's part 2's loop with its counter, in one line.

🧪 **Your turn #1.** How many custody levels are written as nothing at all, the empty string `""`?

In [ ]:
answer_1 = ___      # compare custody with "", then .sum()
check.p3_ex1(answer_1)

## 4. `.str`: part 1's methods, for every value
A column's `.str` gives you all of part 1's string methods, done to every value at once: `custody.str.strip()` strips all 100,000.

In [ ]:
custody.str.strip().value_counts().head(5)     # the five most common values, once stripped

🧪 **Your turn #2.** How many custody levels are empty **once they're stripped**? That's Q1's real answer.

In [ ]:
answer_2 = ___      # strip first, then compare with "" and .sum()
check.p3_ex2(answer_2)

Almost twice as many as #1. Half the missing custody levels were hiding as spaces.

## 5. Filtering: just the rows you want
Put a True/False column inside `.loc[...]` and you get only the rows where it's `True`. After the comma comes a list of the columns you want to see:

In [ ]:
missing = custody.str.strip() == ""
raw.loc[missing, ["offender_number", "custody_level"]].head()

They look empty, but some have spaces in them. A report would count those as having a custody level. Course 1 turned them into real empty values.

## 6. `.mask`: empty wherever it's True
`.mask(...)` makes a value empty (`NaN`, which is how pandas shows `None`) wherever the question is `True`, and leaves the rest alone. `.isna()` asks "is this empty?" of every value:

In [ ]:
stripped = custody.str.strip()
cleaned = stripped.mask(stripped == "")
print("Empty before:", custody.isna().sum())
print("Empty after: ", cleaned.isna().sum())

That's strip-and-null for one column. Course 1 does it for **every** column.

## 7. Write `strip_and_null`
🧪 **Your turn #3.** Finish the function. It loops over every column (part 2's `for`), strips it and makes the empty values null. One line is missing: the stripped column. The check runs it on the whole file and compares your table with course 1's, cell for cell.

In [ ]:
def my_strip_and_null(df):
    """Trim spaces and tabs from both ends of every value, then turn empty text into a real empty value."""
    out = df.copy()
    for col in out.columns:
        stripped = ___      # the column out[col], with .str.strip()
        out[col] = stripped.mask(stripped == "")
    return out

check.p3_ex3(my_strip_and_null)

## 8. The payoff: Q1, from your own table

In [ ]:
check.p3_payoff(my_strip_and_null)

Here is the course 1 function you just wrote:

In [ ]:
inside.show(cleaning.strip_and_null)

It's yours, line for line. The one difference is line 4: course 1 loops over `text_columns(out)`, so the function still works on a table that also has number or date columns. Ours only has text, so `out.columns` did the same job.

Compare it with part 2. There's no counter and no inner loop: pandas does the loop over 100,000 values for you, and that's what most of course 1's code is.

## Wrap-up
- A **DataFrame** is a table and a **Series** is one of its columns: `raw["custody_level"]`.
- `column == value` asks a question of every value; `.sum()` counts the `True`s.
- `.str` gives every value part 1's methods. `.loc[question, columns]` filters rows. `.mask` empties values, and `.isna()` finds them.
- Missing values hide as spaces. Q1's answer doubled once they were stripped.

**Next, part 4 · Patterns:** how course 1 proved every offender number is 8 digits and recognized nine date formats, with regular expressions.

## Answers

**#1**
```python
answer_1 = (custody == "").sum()
```
**#2**
```python
answer_2 = (custody.str.strip() == "").sum()
```
**#3**
```python
stripped = out[col].str.strip()
```